# Privy on a Colab GPU

Runs the white-box analysis (reading a model's activations) on Colab's GPU, and can run the whole web app there too.

**Before you start:** Runtime → Change runtime type → **T4 GPU** (free) or **A100 / L4** (Colab Pro). Then run the cells in order. Open this notebook directly from GitHub: File → Open notebook → GitHub → `dcow12/Privy`.

In [ ]:
# 1. Check the GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Get the code

Clones the project from GitHub (github.com/dcow12/Privy). If that fails, it asks you to upload `privy_colab.zip` instead.

In [ ]:
import os, subprocess, zipfile
from google.colab import files

ROOT = "/content/privy"
if not os.path.exists(f"{ROOT}/app.py"):
    cloned = subprocess.run(["git", "clone", "-q", "https://github.com/dcow12/Privy", ROOT]).returncode == 0
    if not cloned:
        uploaded = files.upload()  # pick privy_colab.zip
        zipfile.ZipFile(next(iter(uploaded))).extractall(ROOT)
%cd {ROOT}
!ls

In [ ]:
# 3. Install (Colab's own CUDA build of PyTorch is kept; this takes 1-2 minutes)
!pip install -q -r requirements.txt

## 4. Keys

Add keys in Colab's **Secrets** panel (the key icon in the left sidebar) and switch on notebook access for each one. Names it looks for: `HF_TOKEN` (needed for Llama and Gemma models), `ANTHROPIC_API_KEY`, `OPENAI_API_KEY`, `OPENROUTER_API_KEY`, `GEMINI_API_KEY`, `NDIF_API_KEY`. None are required for Qwen models or simulated agents.

In [ ]:
from google.colab import userdata

for name in ["HF_TOKEN", "ANTHROPIC_API_KEY", "OPENAI_API_KEY", "OPENROUTER_API_KEY", "GEMINI_API_KEY", "NDIF_API_KEY"]:
    try:
        value = userdata.get(name)
    except Exception:
        value = None  # not added, or notebook access not granted
    if value:
        os.environ[name] = value
        print("loaded", name)

## Option A: run the white-box analysis directly

Picks a model that fits the GPU: Qwen2.5-3B-Instruct on a 16 GB T4, Qwen2.5-7B-Instruct on a 40-80 GB A100. The first run downloads the model (6-15 GB). A quick run takes roughly 10-25 minutes on a T4.

In [ ]:
import torch

mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
MODEL = "Qwen/Qwen2.5-7B-Instruct" if mem_gb > 30 else "Qwen/Qwen2.5-3B-Instruct"
OUT = "runs_cli/" + MODEL.split("/")[-1]
print(f"{torch.cuda.get_device_name(0)}, {mem_gb:.0f} GB -> {MODEL}")

!python privy.py whitebox --backend local --device cuda --model {MODEL} --out {OUT} --quick --batch-size 16

In [ ]:
# Look at the headline figure and numbers
from IPython.display import Image, display
import json
display(Image(f"{OUT}/probe_by_layer.png", width=900))
s = json.load(open(f"{OUT}/summary.json"))
{k: s[k] for k in ["policy_competence_auc", "eval_probe_transfer_auc", "eval_probe_transfer_bow", "eval_probe_transfer_embedding"] if k in s}

## Option B: run the whole web app on the GPU

Starts the app inside Colab and gives you a link. In the app, **Inside the model → This computer** now means this Colab GPU; set the model ID to `Qwen/Qwen2.5-3B-Instruct` (T4) or `Qwen/Qwen2.5-7B-Instruct` (A100). Agent experiments with API keys work there too.

In [ ]:
import subprocess, time
from google.colab import output

server = subprocess.Popen(["python", "app.py"], cwd=ROOT,
                          stdout=open("/content/app.log", "w"), stderr=subprocess.STDOUT)
time.sleep(8)
print("If the link below doesn't load, check /content/app.log.")
output.serve_kernel_port_as_window(8000, path="/")

## Save your results

Colab deletes everything when the session ends, so download before closing. On your Mac, unzip it into the project folder: runs from the app (`runs/`) then show up in your local app's Results tab, and command-line results land in `runs_cli/`.

In [ ]:
!cd {ROOT} && zip -qr /content/privy_results.zip runs runs_cli 2>/dev/null; ls -lh /content/privy_results.zip
files.download("/content/privy_results.zip")